## DATA CLEANING AND PREPROCESSING

### DATA CLEANING contains the following processes :
#### Inspect 
#### Find duplicates
#### Find hidden missing values / invalid values
#### Fix data types
#### Handle missing values appropriately
#### Encode categorical variables
#### Feature engineering
#### Verify the cleaned dataset

In [225]:
import pandas as pd 

In [226]:
df = pd.read_csv(r"C:\Users\RATHICK A\OneDrive\Desktop\data analytics\data\Telco_customer_churn.csv")

In [227]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## inspection 

In [228]:
df.shape 

(7043, 33)

In [229]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

#### From the above info we have concluded that we have 5174 missing values in Churn Reason and also the data type for Total Charges attributes are in object format (text) , which is not good as it may cause error when we calculate , so we have to convert it to numeric data type. And also regarding the churn reason , the missing values can also mean that those records with missing values may belong to customer who have not churned.

### Finding duplicates 

In [230]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
7038    False
7039    False
7040    False
7041    False
7042    False
Length: 7043, dtype: bool

In [231]:
df.duplicated().sum()

np.int64(0)

In [232]:
df["CustomerID"].duplicated().sum()

np.int64(0)

### this says that there is no duplicates 

In [233]:
df["Total Charges"].isna().sum()

np.int64(0)

In [234]:
df["Churn Reason"].isna().sum()

np.int64(5174)

In [235]:
pd.crosstab(df["Churn Label"], df["Churn Reason"].isna())        

Churn Reason,False,True
Churn Label,,
No,0,5174
Yes,1869,0


### 5,174 missing values, all belonging to customers who did not churn. Missingness is expected and semantically meaningful.

In [236]:
df["Total Charges"].value_counts().head(20)

Total Charges
         11
20.2     11
19.75     9
19.9      8
20.05     8
19.65     8
19.55     7
45.3      7
20.15     6
19.45     6
20.25     6
20.3      5
20.45     5
75.3      4
69.65     4
69.6      4
19.5      4
74.7      4
70.6      4
20.4      4
Name: count, dtype: int64

In [237]:
df["Total Charges"].isna().sum()

np.int64(0)

### The empty space before the first row in the above data set says that , 11 values are blank/empty strings . But the above cell says that the number of empty values in the Total Charges attribute is zero . So we should convert the empty string to null value and also convert the data type to numeric from text/string type 

In [238]:
df["Total Charges"] = df["Total Charges"].replace(" ", pd.NA)

In [239]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"])

In [240]:
df["Total Charges"].isna().sum()

np.int64(11)

### Now we have done the above mentioned work.

### Now we have to check why we have null values in the Total Charges . 

In [241]:
df[df["Total Charges"].isna()][
    ["CustomerID", "Tenure Months", "Monthly Charges", "Churn Label"]
]

,CustomerID,Tenure Months,Monthly Charges,Churn Label
2234,4472-LVYGI,0,52.55,No
2438,3115-CZMZD,0,20.25,No
2568,5709-LVOEQ,0,80.85,No
2667,4367-NUYAO,0,25.75,No
2856,1371-DWPAZ,0,56.05,No
4331,7644-OMVMY,0,19.85,No
4687,3213-VVOLG,0,25.35,No
5104,2520-SGTTA,0,20.00,No
5719,2923-ARZLG,0,19.70,No
6772,4075-WKNIU,0,73.35,No


### If a customer has just joined and has zero months of tenure, having no accumulated total charge can be legitimate. So let us make the null values as zero which can be taken from a clear business explanation for the missing total charge.

In [242]:
df["Total Charges"] = df["Total Charges"].fillna(0)

### As we are done with handling missing vlaues, now check for invalid/impossible values.

In [243]:
df.describe()

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,Churn Value,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,2279.734304,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,2266.794470,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,0.000000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,398.550000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,1394.550000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,3786.600000,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,8684.800000,1.000000,100.000000,6500.000000


### The Count column has constant value 1 for all the records. For modeling, it provides no useful information.

### Checking Categorical Values

The unique values of each categorical column were inspected to identify the different categories, check for inconsistencies, and decide the appropriate encoding method.

In [244]:
for col in df.select_dtypes(include="object").columns:
    print("\n", col)
    print(df[col].unique())


 CustomerID
['3668-QPYBK' '9237-HQITU' '9305-CDSKC' ... '2234-XADUH' '4801-JZAZL'
 '3186-AJIEK']

 Country
['United States']

 State
['California']

 City
['Los Angeles' 'Beverly Hills' 'Huntington Park' ... 'Standish' 'Tulelake'
 'Olympic Valley']

 Lat Long
['33.964131, -118.272783' '34.059281, -118.30742' '34.048013, -118.293953'
 ... '40.346634, -120.386422' '41.813521, -121.492666'
 '39.191797, -120.212401']

 Gender
['Male' 'Female']

 Senior Citizen
['No' 'Yes']

 Partner
['No' 'Yes']

 Dependents
['No' 'Yes']

 Phone Service
['Yes' 'No']

 Multiple Lines
['No' 'Yes' 'No phone service']

 Internet Service
['DSL' 'Fiber optic' 'No']

 Online Security
['Yes' 'No' 'No internet service']

 Online Backup
['Yes' 'No' 'No internet service']

 Device Protection
['No' 'Yes' 'No internet service']

 Tech Support
['No' 'Yes' 'No internet service']

 Streaming TV
['No' 'Yes' 'No internet service']

 Streaming Movies
['No' 'Yes' 'No internet service']

 Contract
['Month-to-month' 'Two year'

### Initial Data Inspection

- No duplicate rows or obvious inconsistencies were found.
- `Total Charges` contains 11 blank values and is stored as `object`; these need to be handled before converting it to numeric.
- Missing values in `Churn Reason` are expected for customers who did not churn.
- `Count`, `Country`, and `State` are constant across all records and provide no useful variation.
- Categorical values are consistent, with values such as `No internet service` and `No phone service` having valid meanings.

### Identifying Categorical Columns

Selected all columns with an `object` data type to identify the categorical variables that may require encoding before machine learning.

In [245]:
categorical_cols = df.select_dtypes(include="object").columns
categorical_cols

Index(['CustomerID', 'Country', 'State', 'City', 'Lat Long', 'Gender',
       'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Churn Label', 'Churn Reason'],
      dtype='object')

### Checking Number of Categories

The number of unique categories in each categorical column was checked to determine the appropriate encoding method for each variable.

In [246]:
for col in categorical_cols:
    print(col, ":", df[col].nunique())
    

CustomerID : 7043
Country : 1
State : 1
City : 1129
Lat Long : 1652
Gender : 2
Senior Citizen : 2
Partner : 2
Dependents : 2
Phone Service : 2
Multiple Lines : 3
Internet Service : 3
Online Security : 3
Online Backup : 3
Device Protection : 3
Tech Support : 3
Streaming TV : 3
Streaming Movies : 3
Contract : 3
Paperless Billing : 2
Payment Method : 4
Churn Label : 2
Churn Reason : 20


In [247]:
df_encoded = df.copy()

In [248]:
binary_cols = [
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Phone Service",
    "Paperless Billing"
]

for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({"Yes": 1, "No": 0})

In [249]:
df_encoded["Gender"] = df_encoded["Gender"].map({
    "Male": 1,
    "Female": 0
})

In [250]:
df_encoded["Churn Label"] = df_encoded["Churn Label"].map({
    "Yes": 1,
    "No": 0
})

In [251]:
df_encoded = pd.get_dummies(df_encoded, columns=[
    "Multiple Lines", "Internet Service", "Online Security",
    "Online Backup", "Device Protection", "Tech Support",
    "Streaming TV", "Streaming Movies", "Contract", "Payment Method"
], dtype=int)

In [252]:
df_encoded.shape

(7043, 54)

In [253]:
df_encoded.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Streaming Movies_No,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,Payment Method_Bank transfer (automatic),Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,1,...,1,0,0,1,0,0,0,0,0,1
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,0,...,1,0,0,1,0,0,0,0,1,0
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,0,...,0,0,1,1,0,0,0,0,1,0
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,0,...,0,0,1,1,0,0,0,0,1,0
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,1,...,0,0,1,1,0,0,1,0,0,0


### Encoding

Categorical columns were converted into numerical form so that they can be used effectively for machine learning.

- Binary columns such as `Gender`, `Partner`, `Dependents`, `Phone Service`, and `Paperless Billing` were encoded as `0/1`.
- `Churn Label` was also converted to `0/1` as the target variable.
- Multi-category columns such as `Internet Service`, `Contract`, `Multiple Lines`, and `Payment Method` were converted using one-hot encoding.
- `CustomerID`, `City`, `Lat Long`, and `Churn Reason` were not directly encoded because they are identifiers, high-cardinality/location data, or target-related information.

df_encoded.info()

In [254]:
model_df = df_encoded.drop(columns=[
    "CustomerID",
    "Country",
    "State",
    "City",
    "Lat Long",
    "Churn Reason"
])

### we're creating a cleaner dataset specifically for modeling.

In [255]:
model_df.shape

(7043, 48)

In [256]:
model_df.select_dtypes(include="object").columns

Index([], dtype='object')

In [257]:
model_df.isna().sum().sum()

np.int64(0)

### The previous three cells were checkingof the model dataset.

### Feature Engineering 

In [258]:
model_df["Tenure Group"] = pd.cut(
    model_df["Tenure Months"],
    bins=[-1, 12, 36, 72],
    labels=["New", "Medium", "Long-term"]
)

### Tenure Group

Customers were grouped into New, Medium, and Long-term categories based on their tenure months. This feature helps identify whether customer retention duration has a relationship with churn behavior.

In [259]:
model_df["Total Services"] = (
    (model_df["Phone Service"] == 1).astype(int) +
    (model_df["Multiple Lines_Yes"] == 1).astype(int) +
    (model_df["Online Security_Yes"] == 1).astype(int) +
    (model_df["Online Backup_Yes"] == 1).astype(int) +
    (model_df["Device Protection_Yes"] == 1).astype(int) +
    (model_df["Tech Support_Yes"] == 1).astype(int) +
    (model_df["Streaming TV_Yes"] == 1).astype(int) +
    (model_df["Streaming Movies_Yes"] == 1).astype(int)
)

### Total Services

A `Total Services` feature was created by counting the number of services used by each customer. This helps measure overall customer engagement and analyze whether customers using more services have different churn behavior.

In [260]:
model_df["Support Services"] = (
    (model_df["Online Security_Yes"] == 1).astype(int) +
    (model_df["Online Backup_Yes"] == 1).astype(int) +
    (model_df["Device Protection_Yes"] == 1).astype(int) +
    (model_df["Tech Support_Yes"] == 1).astype(int)
)

### Support Services

A `Support Services` feature was created by combining security, backup, device protection, and technical support services. This helps measure the level of support and protection used by each customer and its possible relationship with churn.

## Final Validation of the processed dataset 

In [261]:
model_df.shape

(7043, 51)

In [262]:
model_df.isna().sum().sum()

np.int64(0)

In [263]:
model_df.dtypes.value_counts()

int64       46
float64      4
category     1
Name: count, dtype: int64

In [264]:
model_df[["Tenure Group", "Total Services", "Support Services"]].head()

,Tenure Group,Total Services,Support Services
0,New,3,2
1,New,1,0
2,New,5,1
3,Medium,6,2
4,Long-term,6,2


In [265]:
[col for col in ["Churn Reason", "Churn Score", "Churn Value", "Churn Label"]
 if col in model_df.columns]

['Churn Score', 'Churn Value', 'Churn Label']

### Target Leakage Check

Checked the churn-related columns to avoid giving the model information about the answer.

- `Churn Label` → kept as the target.
- `Churn Value` → excluded because it directly represents the churn label.
- `Churn Score` → excluded because it is already related to churn.
- `Churn Reason` → excluded because it is only known after a customer churns.

This helps the model learn from actual customer features instead of information that already reveals the outcome.

In [ ]:
to_csv("telco_customer_churn_processed.csv", index=False)

In [269]:
model_df.to_csv("telco_customer_churn_processed_for_modelling.csv", index=False)

In [270]:
import os
os.getcwd()

'C:\\Users\\RATHICK A\\anaconda_projects\\09d752e3-d007-41c9-a9e4-2899988bda05'

In [271]:
os.listdir()

['.ipynb_checkpoints', 'telco_customer_churn_processed_for_modelling.csv']

In [272]:
df.to_csv("telco_customer_churn_processed.csv", index=False)

In [273]:
import os
os.getcwd()

'C:\\Users\\RATHICK A\\anaconda_projects\\09d752e3-d007-41c9-a9e4-2899988bda05'